# Tutorial: End-to-End MPP Analysis

This notebook walks through a complete MPP analysis using the sample dataset
included in the repository (`example/sample_system/`). The dataset contains a
100 000-frame microstate trajectory with 6 microstates and a corresponding
single-feature trajectory.

**Run from the repository root** so that relative paths resolve correctly.

## 1 — Inspect the input data

In [ ]:
import numpy as np

traj = np.loadtxt("example/sample_system/input/traj", dtype=int)
feat = np.loadtxt("example/sample_system/input/feature_traj", ndmin=2)

print(f"Trajectory: {traj.shape[0]} frames, {len(np.unique(traj))} unique microstates")
print(f"Features:   {feat.shape[1]} feature(s) per frame")

## 2 — Run the MPP lumping (Python API)

In [ ]:
import MPP
import MPP.kernel

kernel = MPP.kernel.LumpingKernel(similarity="T")

mpp = MPP.Lumping(
    traj,
    lagtime=20,
    feature_trajectory=feat,
    pop_thr=0.15,
    q_min=0.5,
    frame_length=0.2,   # ns per frame
)
mpp.run_mpp(kernel)

## 3 — Inspect macrostate results

In [ ]:
n = mpp.n_macrostates[0]
print(f"Number of macrostates: {n}")

pop = mpp.macrostate_population[0]
fracs = pop / pop.sum()
for i, (p, f) in enumerate(zip(pop, fracs)):
    print(f"  Macrostate {i}: {p} frames  ({f:.1%})")

print(f"\nMacrostate map (microstate → macrostate): {mpp.macrostate_map[0]}")

## 4 — Quality metrics

In [ ]:
print(f"Shannon entropy:    {mpp.shannon_entropy[0]:.4f}  (0 = single state, 1 = uniform)")
print(f"Davies-Bouldin:     {mpp.davies_bouldin_index[0]:.4f}  (lower = better separated)")
print(f"GMRQ:               {mpp.gmrq[0]:.4f}  (higher = better slow-dynamics preservation)")
print(f"Silhouette:         {mpp.silhouette[0]:.4f}  (-1 to 1; higher = better)")
print(f"Calinski-Harabász:  {mpp.calinski_harabasz[0]:.1f}  (higher = better)")

## 5 — Generate plots

In [ ]:
import os
os.makedirs("results/t", exist_ok=True)

mpp.plot.dendrogram("results/t/dendrogram.pdf")
mpp.plot.macrostate_trajectory("results/t/macrotraj.pdf")
mpp.plot.ck_test("results/t/ck_test.pdf")
mpp.plot.state_network("results/t/state_network.pdf")
mpp.plot.transition_matrix("results/t/transition_matrix.pdf")
print("Plots saved to results/t/")

## 6 — Save the macrostate trajectory

In [ ]:
mpp.save_macrostate_trajectory("results/t/macrostate_trajectory.txt", one_based=True)

saved = np.loadtxt("results/t/macrostate_trajectory.txt", dtype=int)
print(f"Saved {len(saved)} macrostate assignments, values: {np.unique(saved)}")

## 7 — Same workflow via CLI

All of the above can also be run from the command line. The `-Z` flag saves or
loads the lumping tree so the expensive computation is not repeated.

In [ ]:
import subprocess

base = ["python", "-m", "MPP.run",
        "example/sample_system/input/config.yml", "T", "none",
        "-Z", "results/t/Z.npy"]

# Run lumping (or load existing Z matrix)
subprocess.run(base, check=True)

# Print quality metrics
result = subprocess.run(base + ["--metrics"], capture_output=True, text=True)
print(result.stdout)

## 8 — Try other kernels

In [ ]:
# Kullback-Leibler divergence kernel
kernel_kl = MPP.kernel.LumpingKernel(similarity="KL")
mpp_kl = MPP.Lumping(traj, lagtime=20, feature_trajectory=feat,
                     pop_thr=0.15, q_min=0.5, frame_length=0.2)
mpp_kl.run_mpp(kernel_kl)
print(f"KL lumping: {mpp_kl.n_macrostates[0]} macrostates")

# Combined T + Jensen-Shannon feature similarity
kernel_t = MPP.kernel.LumpingKernel(similarity="T")
feat_kernel = MPP.kernel.FeatureKernel(feat, traj)
mpp_tjs = MPP.Lumping(traj, lagtime=20, feature_trajectory=feat,
                      pop_thr=0.15, q_min=0.5, frame_length=0.2)
mpp_tjs.run_mpp(kernel_t, feature_kernel=feat_kernel)
print(f"T+JS lumping: {mpp_tjs.n_macrostates[0]} macrostates")

## 9 — Config-based workflow

`MPP.run.Data` reads the YAML config and orchestrates the full pipeline,
matching the CLI behaviour exactly.

In [ ]:
from MPP.run import Data

data = Data("example/sample_system/input/config.yml")
data.setup_mpp("T", "none")
data.perform_mpp("results/t/Z.npy")   # loads existing Z matrix

mpp2 = data.mpp
print(f"Macrostates: {mpp2.n_macrostates[0]}")
print(f"Shannon entropy: {mpp2.shannon_entropy[0]:.4f}")